In [ ]:
# Dev kernel: smoke-test QwenAgent on Kaggle's H100 with the bundled
# Qwen3.6-35B-A3B BF16 weights mounted via dataset_sources.
#
# Pre-reqs:
#   1. Bundler kernel (exp004_qwen_agent/bundle_qwen_kernel) ran COMPLETE.
#   2. Private Kaggle Dataset 'cataluna84/qwen3-6-35b-a3b-bf16' exists.
#   3. competition data attached (gives us /kaggle/input/arc-prize-2026-arc-agi-3/).
#
# This kernel only smoke-tests the loop -- it does NOT write a submission.
# Promotion to a competition kernel happens after the smoke results look healthy.
import os, sys, time, json, subprocess
from pathlib import Path

# --- 1. Locate the mounted Qwen weights -------------------------------------
# Kaggle datasets used to mount at /kaggle/input/<slug>/ but newer kernel
# images use a nested layout: /kaggle/input/datasets/<owner>/<slug>/. We try
# the legacy path first, then the nested path, then fall back to a recursive
# scan for model.safetensors.index.json.
_CANDIDATES = [
    Path('/kaggle/input/qwen3-6-35b-a3b-bf16'),
    Path('/kaggle/input/datasets/cataluna84/qwen3-6-35b-a3b-bf16'),
]
QWEN_DIR = next((c for c in _CANDIDATES if c.exists() and (c / 'model.safetensors.index.json').exists()), None)
if QWEN_DIR is None:
    for p in Path('/kaggle/input').rglob('model.safetensors.index.json'):
        QWEN_DIR = p.parent
        break
if QWEN_DIR is None:
    print('[ERROR] Qwen dataset not mounted (looked in legacy + nested + recursive)')
    print('Tree of /kaggle/input/ (depth 3):')
    import subprocess as _sp
    _sp.run(['find', '/kaggle/input', '-maxdepth', '3', '-type', 'd'])
    sys.exit(1)

print('Qwen weights at', QWEN_DIR)
print('Total size:', sum(f.stat().st_size for f in QWEN_DIR.rglob('*') if f.is_file())/1e9, 'GB')
for f in sorted(QWEN_DIR.rglob('*'))[:10]:
    if f.is_file():
        print(f'  {f.stat().st_size/1e9:>6.2f} GB  {f.relative_to(QWEN_DIR)}')

# --- 2. Install the arc-agi SDK from the competition wheels ----------------
# The SDK is needed for the real env loop. The wheels are bundled under the
# competition data at /kaggle/input/arc-prize-2026-arc-agi-3/.
_WHEEL_CANDS = [
    Path('/kaggle/input/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'),
    Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'),
]
WHEELS = next((c for c in _WHEEL_CANDS if c.exists()), _WHEEL_CANDS[0])
if WHEELS.exists():
    print('installing pillow + arc-agi + arcengine from', WHEELS)
    # The Kaggle H100 image ships PIL 11.3.0 which has a packaging bug:
    # PIL/ImageText.py imports `_Ink` from PIL/_typing.py but _Ink is missing.
    # Worse, the C extension `_imaging.cpython-312-x86_64-linux-gnu.so` is
    # baked into the system image at a path pip cannot replace cleanly.
    # Strategy: install pillow-12.2.0 to a private --target dir and prepend
    # that to sys.path so `import PIL.*` resolves there before reaching
    # /usr/local/.
    PILLOW_TARGET = '/kaggle/working/_pillow_pkg'
    Path(PILLOW_TARGET).mkdir(parents=True, exist_ok=True)
    r2 = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
                         f'--find-links={WHEELS}', '--upgrade',
                         '--target', PILLOW_TARGET, 'pillow'],
                        capture_output=True, text=True)
    print('  pip install pillow stdout:', r2.stdout[-500:])
    print('  pip install pillow stderr:', r2.stderr[-200:])
    # Prepend the private dir to sys.path so it's searched FIRST.
    if PILLOW_TARGET not in sys.path:
        sys.path.insert(0, PILLOW_TARGET)
    # Purge any stale PIL modules so the next import re-resolves from the
    # new location.
    for mod in [m for m in list(sys.modules) if m.startswith('PIL')]:
        del sys.modules[mod]
    try:
        import PIL, PIL.ImageText  # noqa: F401
        print(f'  PIL ok: {PIL.__version__} at {PIL.__file__}')
    except Exception as e:
        print(f'  [WARN] PIL still broken: {type(e).__name__}: {e}')
    # Now install arc-agi + arcengine into the system path (their deps are
    # all already on the Kaggle image; only the SDK itself is custom).
    r3 = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                         '--no-index', f'--find-links={WHEELS}',
                         'arc-agi', 'arcengine'],
                        capture_output=True, text=True)
    print('  pip install arc-agi stdout:', r3.stdout[-300:])
    print('  pip install arc-agi stderr:', r3.stderr[-300:])
else:
    print('[WARN] no wheels at', WHEELS, '- skipping SDK install')

# --- 2b. Install transformers 5.7.0 from our offline-mirror Dataset ---------
# Kaggle's H100 image ships transformers 5.0.0 which does NOT recognize the
# `qwen3_5_moe` model_type used by Qwen3.6-35B-A3B. We bundled transformers
# 5.7.0 + minimal deps as a separate Kaggle Dataset; install into a private
# --target dir (same trick as PIL) and prepend to sys.path so the new
# transformers wins import resolution over the image's pre-installed one.
_TX_CANDS = [
    Path('/kaggle/input/arc-agi-3-transformers-wheels'),
    Path('/kaggle/input/datasets/cataluna84/arc-agi-3-transformers-wheels'),
]
TX_WHEELS = next((c for c in _TX_CANDS if c.exists()), None)
if TX_WHEELS is None:
    for p in Path('/kaggle/input').rglob('transformers-*.whl'):
        TX_WHEELS = p.parent
        break
if TX_WHEELS is not None:
    TX_TARGET = '/kaggle/working/_transformers_pkg'
    Path(TX_TARGET).mkdir(parents=True, exist_ok=True)
    print(f'installing transformers + deps from {TX_WHEELS} -> {TX_TARGET}')
    # --no-deps because numpy/torch/PIL/etc. are all in the Kaggle image
    # already; we only need to *replace* transformers + tokenizers + the
    # huggingface stack with the newer versions.
    r5 = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
                         f'--find-links={TX_WHEELS}', '--upgrade',
                         '--target', TX_TARGET, '--no-deps',
                         'transformers', 'tokenizers', 'accelerate',
                         'huggingface_hub', 'safetensors',
                         'regex', 'filelock', 'fsspec', 'pyyaml', 'tqdm'],
                        capture_output=True, text=True)
    print('  pip install transformers stdout:', r5.stdout[-1000:])
    print('  pip install transformers stderr:', r5.stderr[-500:])
    # Confirm files actually landed in TX_TARGET, then prepend.
    print(f'  contents of {TX_TARGET} (top-level):')
    for p in sorted(Path(TX_TARGET).iterdir()):
        print(f'    {p.name}')
    if TX_TARGET not in sys.path:
        sys.path.insert(0, TX_TARGET)
    for mod in [m for m in list(sys.modules) if m.split('.')[0] in {
        'transformers', 'tokenizers', 'accelerate', 'huggingface_hub',
        'safetensors'}]:
        del sys.modules[mod]
    try:
        import transformers as _tx
        print(f'  transformers ok: {_tx.__version__} at {_tx.__file__}')
        if not _tx.__file__.startswith(TX_TARGET):
            print(f'  [WARN] transformers loaded from system path, not target!')
    except Exception as e:
        print(f'  [WARN] transformers import failed: {type(e).__name__}: {e}')
else:
    print('[WARN] transformers wheels Dataset not mounted - using image default')

# --- 3. Make our local agents/ package importable --------------------------
# We attach this kernel to a kernel_sources entry that points at our agents
# repo, OR we paste the repo into /kaggle/working at kernel-edit time.
# For v1 we paste the relevant agent files into the kernel via 'kernel_sources'.
AGENTS_PKG = Path('/kaggle/working/agents')
if not AGENTS_PKG.exists():
    _AGENT_CANDS = [
        Path('/kaggle/input/arc-agi-3-agents-pkg/agents'),
        Path('/kaggle/input/datasets/cataluna84/arc-agi-3-agents-pkg/agents'),
    ]
    src = next((c for c in _AGENT_CANDS if c.exists()), None)
    if src is None:
        for p in Path('/kaggle/input').rglob('qwen_agent.py'):
            src = p.parent
            break
    if src is not None:
        import shutil
        shutil.copytree(src, AGENTS_PKG)
        print(f'copied agents pkg from {src} -> {AGENTS_PKG}')
    else:
        print('[WARN] agents pkg not found - QwenAgent import will fail')
sys.path.insert(0, '/kaggle/working')

# --- 4. Configure the agent and run on one game ----------------------------
os.environ['QWEN_MODEL_PATH'] = str(QWEN_DIR)
os.environ['QWEN_DTYPE']      = 'bf16'
os.environ['QWEN_DEVICE_MAP'] = 'auto'
os.environ['QWEN_DEBUG_PROMPTS'] = '1'
os.environ['QWEN_MAX_NEW_TOKENS'] = '96'

# arc-agi SDK runs in OFFLINE mode (no internet on the eval box). Point it
# at the mounted competition data's environment_files dir.
os.environ['OPERATION_MODE'] = 'offline'
_ENV_CANDS = [
    Path('/kaggle/input/arc-prize-2026-arc-agi-3/environment_files'),
    Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files'),
]
ENV_DIR = next((c for c in _ENV_CANDS if c.exists()), None)
if ENV_DIR is None:
    print('[WARN] environment_files not found in expected paths; recursive search:')
    for p in Path('/kaggle/input').rglob('environment_files'):
        if p.is_dir():
            ENV_DIR = p
            print(f'  found: {p}')
            break
if ENV_DIR is not None:
    os.environ['ENVIRONMENTS_DIR'] = str(ENV_DIR)
    print(f'ENVIRONMENTS_DIR = {ENV_DIR}')
    print(f'  contains {sum(1 for _ in ENV_DIR.iterdir())} game dirs')

from agents.qwen_agent import QwenAgent  # noqa: E402
from agents import GameAction, GameState  # noqa: E402

GAME_ID = os.environ.get('SMOKE_GAME', 'ls20')
MAX_ACTIONS = int(os.environ.get('MAX_ACTIONS', 50))

from arc_agi import Arcade, OperationMode  # noqa: E402
arcade = Arcade(operation_mode=OperationMode.OFFLINE,
                environments_dir=os.environ.get('ENVIRONMENTS_DIR', 'environment_files'))
env = arcade.make(GAME_ID)
frame = env.observation_space

agent = QwenAgent(arc_env=env, game_id=GAME_ID)
print(f'starting smoke run on {GAME_ID}, max_actions={MAX_ACTIONS}')

history_log = []
t0 = time.time()
for step in range(MAX_ACTIONS):
    if frame.state in (GameState.WIN, GameState.GAME_OVER):
        print(f'  [step {step}] terminal state {frame.state}, breaking')
        break
    ts = time.time()
    action = agent.choose_action(frame)
    dt = time.time() - ts
    print(f'  [step {step}] {action.name}  state={frame.state}  '
          f'level={frame.levels_completed}  dt={dt:.2f}s')
    history_log.append({'step': step, 'action': action.name,
                        'state': str(frame.state), 'dt': round(dt,3)})
    data = {}
    ad = getattr(action, 'action_data', None)
    if ad is not None:
        try: data = ad.model_dump()
        except Exception: data = getattr(action, '_data', {}) or {}
    next_frame = env.step(action, data=data, reasoning=None)
    if next_frame is None:
        print('  env.step returned None')
        break
    frame = next_frame

elapsed = time.time() - t0
print()
print('=== SMOKE SUMMARY ===')
print(f'  game             : {GAME_ID}')
print(f'  actions taken    : {len(history_log)}')
print(f'  levels_completed : {getattr(frame, "levels_completed", 0)}')
print(f'  win_levels       : {getattr(frame, "win_levels", 0)}')
print(f'  final state      : {frame.state}')
print(f'  wall clock       : {elapsed:.1f}s ({elapsed/max(len(history_log),1):.2f}s/action)')

with open('/kaggle/working/qwen_smoke.json', 'w') as fh:
    json.dump({
        'game': GAME_ID,
        'actions': history_log,
        'levels_completed': getattr(frame, 'levels_completed', 0),
        'final_state': str(frame.state),
        'wall_clock_s': round(elapsed, 2),
    }, fh, indent=2)
print('saved /kaggle/working/qwen_smoke.json')
